# Imports

In [ ]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from scipy.special import logit, expit

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, cross_val_predict

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

def get_logit(proba, eps=1e-15):
    proba = np.clip(proba, eps, 1 - eps)
    return logit(proba)

# Loading Dataset

In [4]:
X_train_proba = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
X_test_proba = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [5]:
X_train_proba.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_logit,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.500714,0.868510,0.716126,0.773266
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.335105,0.772514,0.636769,0.591885
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.136991,0.717012,0.546431,0.516648
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.247050,0.005494,0.112073,0.001113
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.094611,0.482416,0.445065,0.293469


In [6]:
X_test_proba.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_logit,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,0.019849,-0.629264,0.006334
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,0.012173,-0.279447,0.003675
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,0.010304,-0.661186,0.003484
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.444558,0.424415,0.238053
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.939648,0.688728,0.895339


# Feature Engineering

## Transform to Logit

In [19]:
to_logit_columns = [col for col in X_train_proba.columns.tolist() if col != 'ridge_logit']

In [17]:
X_train_logit = X_train_proba.copy()
X_test_logit = X_test_proba.copy()

In [20]:
X_train_logit[to_logit_columns] = X_train_logit[to_logit_columns].apply(get_logit)
X_test_logit[to_logit_columns] = X_test_logit[to_logit_columns].apply(get_logit)

In [36]:
X_train_logit = X_train_logit.rename(columns=lambda col: col.replace('proba', 'logit'))
X_test_logit = X_test_logit.rename(columns=lambda col: col.replace('proba', 'logit'))

# Saving

In [37]:
X_test_logit.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,-5.379032,-3.409957,-3.759100,-3.888929,-5.042109,-34.538776,-1.272709,-3.899528,-34.538776,-5.055490
1,-5.795381,-4.426299,-4.187473,-3.950213,-2.684867,-2.019087,-0.626960,-4.396275,-34.538776,-5.602505
2,-5.574937,-4.441487,-4.232580,-4.454119,-5.395432,-34.538776,-1.335019,-4.564839,-34.538776,-5.656104
3,-1.635113,0.176618,0.347033,-0.058463,0.644945,-2.198160,0.190133,-0.222683,-0.304675,-1.163384
4,1.920299,3.398116,3.163860,3.161400,1.661579,34.539576,0.535153,2.745303,0.794179,2.146481


In [38]:
X_train_logit.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,1.402666,2.109990,2.149899,2.030346,1.440273,2.064999,0.500714,1.887848,0.925326,1.226844
1,0.616425,1.595299,1.584592,1.262993,0.799597,0.079907,0.335105,1.222563,0.561368,0.371762
2,0.023615,1.748857,1.106745,1.146720,0.235157,-1.325798,0.136991,0.929687,0.186260,0.066618
3,-6.367775,-5.125195,-4.946636,-4.912857,-6.083520,-34.538776,-1.247050,-5.198584,-2.069743,-6.799934
4,-0.438053,0.263538,-0.043875,-0.068158,-0.346862,-0.861416,-0.094611,-0.070366,-0.220629,-0.878595


In [39]:
X_train_logit.to_parquet('../data/processed/X_train_logit.parquet')
X_test_logit.to_parquet('../data/processed/X_test_logit.parquet')